# GPU stage on Colab: BERT dense retrieval and learning-to-rank fine-tuning

Runtime -> Change runtime type -> **T4 GPU**. This notebook is a thin wrapper around `run_gpu.sh`; all logic is in `scripts/`.

Only `google-bert/bert-base-uncased` at its verified revision can be loaded (BookCorpus + English Wikipedia, MLM/NSP; see `docs/model_eligibility.md`). The scripts refuse any other model.

Required on Drive at `MyDrive/DATA304_final_project`:
- the latest `scripts/`, `src/`, `run_gpu.sh`
- `project_release/` (`catalog.json`, `train_gold.json`, `test_queries.json`)
- `outputs/ltr/negatives_dev.json` and `outputs/ltr/negatives_full.json` (made locally by `scripts/mine_negatives.py`, BM25 only, item IDs only). If they are missing, `run_gpu.sh` mines them here, which is slow on Colab's 2 CPUs.

No review text is sent to any LLM service. Every step writes to Drive and finished steps are skipped, so after a disconnect just re-run the same cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/DATA304_final_project'
%cd $PROJECT
!ls scripts src project_release outputs/ltr

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.40"
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## Step 1: zero-shot BERT embeddings (dense retrieval baseline), about 20 minutes

In [ ]:
!bash run_gpu.sh zeroshot

## Step 2: LTR model A (12,000 queries; epoch selected on the 4,000-query tuning slice) + its embeddings, about 1.5 hours

In [ ]:
!bash run_gpu.sh ltr-dev
import json
log = json.load(open('outputs/ltr/bert-ltr-dev/train_log.json'))
for e in log['epochs']: print(e)
print('selected epoch:', log['selected_epoch'])

## Step 3: LTR model B (all 20,000 queries, stops at model A's epoch) + test embeddings, about 1 hour

In [ ]:
!bash run_gpu.sh ltr-full
!ls -la outputs/dense/bert-base-uncased outputs/dense/bert-ltr-dev outputs/dense/bert-ltr-full

Download into the same paths in the local project:
- `outputs/dense/bert-base-uncased/` (catalog.npy, train_reviews.npy, test_reviews.npy, manifest.json)
- `outputs/dense/bert-ltr-dev/` (catalog.npy, train_reviews.npy, manifest.json)
- `outputs/dense/bert-ltr-full/` (catalog.npy, test_reviews.npy, manifest.json)
- `outputs/ltr/bert-ltr-dev/train_log.json` and `outputs/ltr/bert-ltr-full/train_log.json`

Keep the `checkpoint/` folders on Drive; the README will link to them for reproduction.